In [ ]:
import json
import os
import paramiko
import re
import shlex
import subprocess
import sys
import time
from pathlib import Path

from IPython.display import clear_output


# Paths & Cluster Config

Edit the paths below to match your cluster setup before running.

In [ ]:
# Local repo root on this Mac. Edit this if the checkout moves.
LOCAL_PIPELINE_WORKDIR = Path('/Users/qixinyang/Documents/AdamLab/miniVI_Renana_Pipeline')
LOCAL_ANALYSIS_ROOT = LOCAL_PIPELINE_WORKDIR / 'miniVI_PlaceCell_analysis_V4'
LOCAL_DATA_ROOT = LOCAL_ANALYSIS_ROOT / 'data'
LOCAL_REFINED_INPUT_BUNDLE = LOCAL_ANALYSIS_ROOT / 'cluster_exports' / 'ckii_refined_cluster_input_v1.pkl'
LOCAL_PYTHON = sys.executable

# Local bundle build controls.
BUILD_REFINED_INPUT_BUNDLE = True
DRY_RUN_REFINED_EXPORT_FIRST = True
REBUILD_REFINED_PAYLOADS = True
FORCE_REFINED_EXPORT = True

# Cluster login.
CLUSTER_HOST = 'loginserver.elsc.huji.ac.il'
USERNAME = 'qixin.yang'
SSH_KEY_PATH = os.path.expanduser('~/.ssh/id_rsa')  # set to None to use SSH agent

# Repo root on the cluster.
PIPELINE_WORKDIR = '/ems/elsc-labs/adam-y/qixin.yang/ClusterCode/miniVI_Renana_Pipeline'
sh_script = f'{PIPELINE_WORKDIR}/utils/run_python_job.sh'
CONDA_ENV = 'adamlab_pipeline'  # Python 3.11

# Shared data root on the cluster. Large pickles belong here, not under ClusterCode.
SHARED_DATA_ROOT = '/ems/elsc-labs/adam-y/Adam-Lab-Shared/Data/renana_malka'

# Analysis paths on the cluster.
ANALYSIS_ROOT = f'{PIPELINE_WORKDIR}/miniVI_PlaceCell_analysis_V4'
DATA_ROOT     = f'{ANALYSIS_ROOT}/data'
FIGURES_ROOT  = f'{ANALYSIS_ROOT}/figures'
LOG_DIR       = f'{ANALYSIS_ROOT}/logs'
REFINED_INPUT_BUNDLE = f'{SHARED_DATA_ROOT}/cluster_exports/ckii_refined_cluster_input_v1.pkl'
MOUNTED_REFINED_INPUT_BUNDLE = Path(REFINED_INPUT_BUNDLE.replace('/ems/elsc-labs/adam-y', '/Volumes/adam-lab'))

# ===== Analysis parameters (change these between runs) =====
DIRECTION_MODE  = 'head'         # 'head' or 'travel'
SPIKE_TYPE      = 'all_spike'    # 'all_spike', 'simple_spike', or 'complex_spike'
N_SURROGATES    = 1000
FIRST_N_MINUTES = 10.0
FORCE_RECOMPUTE = False
FORCE_UNPACK    = True

# Shared manifest directory (same for all direction/spike combos).
# Only need to run Step 1 once; the manifest is reused across runs.
MANIFEST_DIR = f'{FIGURES_ROOT}/CKII_pooled/egocentric_tuning_carpenter'

# Output directory: encodes direction + spike type so results do not overwrite.
# e.g. .../egocentric_tuning_carpenter/head_all_spike/
OUTPUT_DIR = f'{MANIFEST_DIR}/{DIRECTION_MODE}_{SPIKE_TYPE}'

# Per-cell job resources.
CELL_CPUS   = 10       # CPUs per cell task (for surrogate parallelism)
CELL_MEM_GB = 32       # memory per cell task

# Script paths (relative to PIPELINE_WORKDIR).
PREPARE_SCRIPT   = 'miniVI_PlaceCell_analysis_V4/notebooks_HPC/run_egocentric_prepare.py'
CELL_SCRIPT      = 'miniVI_PlaceCell_analysis_V4/notebooks_HPC/run_egocentric_single_cell.py'
PLOT_SCRIPT      = 'miniVI_PlaceCell_analysis_V4/notebooks_HPC/run_egocentric_plot_cells.py'
STATS_SCRIPT     = 'miniVI_PlaceCell_analysis_V4/notebooks_HPC/run_egocentric_summary_stats.py'

print(f'Local repo:      {LOCAL_PIPELINE_WORKDIR}')
print(f'Local Python:    {LOCAL_PYTHON}')
print(f'Local bundle:    {LOCAL_REFINED_INPUT_BUNDLE}')
print(f'Shared data root:{SHARED_DATA_ROOT}')
print(f'Cluster bundle:  {REFINED_INPUT_BUNDLE}')
print(f'Mounted bundle:  {MOUNTED_REFINED_INPUT_BUNDLE}')
print(f'Run config:      {DIRECTION_MODE} direction, {SPIKE_TYPE} spikes')
print(f'Conda env:       {CONDA_ENV}')
print(f'Manifest dir:    {MANIFEST_DIR}')
print(f'Output dir:      {OUTPUT_DIR}')


# Local Step A: Build Refined Egocentric Cluster Input Bundle

Run this on the local machine before connecting to the cluster. It builds one all-animal pickle from `manual_spike_detection_results.pkl`, `merged_aligned_data_new.pkl`, and the local refined `spatial_analysis_full.pkl` files. The default export is egocentric-only for Step 2 EB shuffling: behavior position/head-direction arrays, spike-frame lists, precomputed bad-frame masks, and reduced place-cell classification metadata. It intentionally excludes traces. Set `BUILD_REFINED_INPUT_BUNDLE = False` if the bundle already exists and you only want to print copy instructions.


In [ ]:
def run_local_command(cmd, *, cwd=LOCAL_PIPELINE_WORKDIR):
    cmd = [str(part) for part in cmd]
    print('[LOCAL]', ' '.join(shlex.quote(part) for part in cmd))
    result = subprocess.run(
        cmd,
        cwd=str(cwd),
        text=True,
        capture_output=True,
        check=False,
    )
    if result.stdout:
        print(result.stdout)
    if result.stderr:
        print(result.stderr)
    if result.returncode != 0:
        raise RuntimeError(f'Local command failed with exit code {result.returncode}')
    return result

export_script = LOCAL_PIPELINE_WORKDIR / 'miniVI_PlaceCell_analysis_V4' / 'notebooks_HPC' / 'export_refined_cluster_input.py'
base_export_cmd = [
    LOCAL_PYTHON,
    export_script,
    '--data-root', LOCAL_DATA_ROOT,
    '--output', LOCAL_REFINED_INPUT_BUNDLE,
]
if REBUILD_REFINED_PAYLOADS:
    base_export_cmd.append('--rebuild-refined')

if DRY_RUN_REFINED_EXPORT_FIRST:
    run_local_command(base_export_cmd + ['--dry-run'])

if BUILD_REFINED_INPUT_BUNDLE:
    export_cmd = list(base_export_cmd)
    if FORCE_REFINED_EXPORT:
        export_cmd.append('--force')
    run_local_command(export_cmd)
    size_gb = LOCAL_REFINED_INPUT_BUNDLE.stat().st_size / (1024 ** 3)
    print(f'[INFO] Wrote {LOCAL_REFINED_INPUT_BUNDLE} ({size_gb:.2f} GB)')
else:
    print('[INFO] Skipped local bundle build because BUILD_REFINED_INPUT_BUNDLE=False')
    if not LOCAL_REFINED_INPUT_BUNDLE.exists():
        raise FileNotFoundError(f'Missing local bundle: {LOCAL_REFINED_INPUT_BUNDLE}')


# Local Step B: Manual Copy Instructions

Run this after building the egocentric-only bundle. It prints the exact local file path and the shared-data destination for manual copy. After copying, connect to the cluster and run Step 0 to unpack it into per-animal runtime files.


In [ ]:
def _quote_path(path):
    return shlex.quote(str(path))

local_bundle = LOCAL_REFINED_INPUT_BUNDLE
cluster_bundle = REFINED_INPUT_BUNDLE
mounted_bundle = MOUNTED_REFINED_INPUT_BUNDLE

print('Copy this local file:')
print(f'  {local_bundle}')
if local_bundle.exists():
    size_gb = local_bundle.stat().st_size / (1024 ** 3)
    print(f'  size: {size_gb:.2f} GB')
else:
    print('  [MISSING] Run Local Step A first to generate this file.')

print()
print('Put it on the cluster at this exact path:')
print(f'  {cluster_bundle}')

print()
print('If the cluster filesystem is mounted on this Mac, copy to:')
print(f'  {mounted_bundle}')
print()
print('Mounted-volume copy commands:')
print(f'mkdir -p {_quote_path(mounted_bundle.parent)}')
print(f'cp {_quote_path(local_bundle)} {_quote_path(mounted_bundle)}')

print()
print('Alternatively, copy with scp from a terminal:')
print(f"ssh {USERNAME}@{CLUSTER_HOST} {_quote_path('mkdir -p ' + os.path.dirname(cluster_bundle))}")
print(f'scp {_quote_path(local_bundle)} {_quote_path(f"{USERNAME}@{CLUSTER_HOST}:{cluster_bundle}")}')

print()
print('After copying, run the Connect cell and then Step 0 unpack.')


# Connect to Cluster

In [3]:
ssh = paramiko.SSHClient()
ssh.set_missing_host_key_policy(paramiko.AutoAddPolicy())
kwargs = {}
if SSH_KEY_PATH and os.path.exists(SSH_KEY_PATH):
    kwargs['key_filename'] = SSH_KEY_PATH
ssh.connect(CLUSTER_HOST, username=USERNAME, **kwargs)
print('[INFO] Connected to cluster.')

def to_local(cluster_path):
    """Convert cluster path to local Mac path via mounted network volume."""
    return cluster_path.replace('/ems/elsc-labs/adam-y', '/Volumes/adam-lab')

def run_command(command):
    """Run a command on the cluster with a login shell."""
    stdin, stdout, stderr = ssh.exec_command(f"bash -l -c '{command}'")
    output = stdout.read().decode().strip()
    error  = stderr.read().decode().strip()
    return output, error

def wait_for_jobs(job_ids, poll_interval=60):
    """Poll SLURM until all jobs complete. Returns True if all succeeded."""
    if not job_ids:
        print('No jobs to wait for.')
        return True
    pending_jobs = set(job_ids)
    failed_jobs = []
    while pending_jobs:
        job_list = ','.join(pending_jobs)
        output, error = run_command(f'sacct -j {job_list} --format=JobID,State,ExitCode -n -P')
        if error and 'Invalid job id' not in error:
            print(f'[WARNING] {error}')
        completed = set()
        for line in output.strip().splitlines():
            if not line or '.' in line.split('|')[0]:
                continue
            parts = line.split('|')
            if len(parts) >= 2:
                job_id, state = parts[0], parts[1]
                if state in ['COMPLETED', 'FAILED', 'CANCELLED', 'TIMEOUT']:
                    completed.add(job_id)
                    if state != 'COMPLETED':
                        failed_jobs.append((job_id, state))
        pending_jobs -= completed
        clear_output(wait=True)
        print(f'[{time.strftime("%H:%M:%S")}] Jobs status:')
        print(f'  Completed: {len(job_ids) - len(pending_jobs)}/{len(job_ids)}')
        print(f'  Pending:   {len(pending_jobs)}')
        if failed_jobs:
            print(f'  Failed:    {failed_jobs}')
        if pending_jobs:
            print(f'\nWaiting {poll_interval}s before next check...')
            time.sleep(poll_interval)
    print('\n' + '=' * 50)
    if failed_jobs:
        print(f'WARNING: {len(failed_jobs)} job(s) failed: {failed_jobs}')
        return False
    print('All jobs completed successfully!')
    return True

[INFO] Connected to cluster.


# Step 0: Unpack Refined Cluster Input (run once)

Assumes the all-animal pickle has already been uploaded to `REFINED_INPUT_BUNDLE` on the cluster shared-data folder. This submits the unpack job directly with `--force` by default, because this workflow needs to replace old cluster `spatial_analysis_full.pkl` files with the reduced spatial classification files from the refined local bundle. The unpack job writes `cluster_refined_analysis_data.pkl`, reduced `spatial_analysis_full.pkl`, and metadata files under `DATA_ROOT/ANIMAL_NAME/`. Set `FORCE_UNPACK = False` only if you want the unpacker to refuse overwriting existing files.


In [ ]:
setup_script = 'miniVI_PlaceCell_analysis_V4/notebooks_HPC/unpack_refined_cluster_input.py'
force_unpack_flag = '--force' if FORCE_UNPACK else ''
setup_job_id = None

print(f'[INFO] Using uploaded bundle on cluster: {REFINED_INPUT_BUNDLE}')
print(f'[INFO] Unpacking into data root: {DATA_ROOT}')

_, mkdir_err = run_command(f'mkdir -p {LOG_DIR}')
if mkdir_err:
    print(f'[WARN] mkdir: {mkdir_err}')

cmd = (
    f'sbatch --job-name ego_setup -c 1 --mem=4G -t 00:05:00 '
    f'--export=ALL,CONDA_ENV_NAME={CONDA_ENV} '
    f'--chdir {PIPELINE_WORKDIR} '
    f'--output {LOG_DIR}/%x_%j.out --error {LOG_DIR}/%x_%j.err '
    f'{sh_script} {PIPELINE_WORKDIR} {setup_script} {REFINED_INPUT_BUNDLE} '
    f'--data-root {DATA_ROOT} {force_unpack_flag}'
)

output, error = run_command(cmd)
if error:
    print(f'[ERROR] {error}')
else:
    print(f'[INFO] {output}')
    match = re.search(r'Submitted batch job (\d+)', output)
    if match:
        setup_job_id = match.group(1)
        print(f'[INFO] Setup Job ID: {setup_job_id}')
        print(f'[INFO] Log .out: {to_local(LOG_DIR)}/ego_setup_{setup_job_id}.out')
        print(f'[INFO] Log .err: {to_local(LOG_DIR)}/ego_setup_{setup_job_id}.err')

if setup_job_id:
    wait_for_jobs([setup_job_id], poll_interval=15)
else:
    print('[INFO] No setup job submitted.')


# Step 1: Prepare — Build Cache & Cell Manifest (run once)

Ensures per-animal cache exists, classifies cells, and writes a `manifest.json`
listing all cells to analyze. The manifest is saved to the **shared** `MANIFEST_DIR`
(not the direction/spike-specific output dir), so you only need to run this once —
then re-run Steps 2–3 with different `DIRECTION_MODE` / `SPIKE_TYPE`.

In [5]:
_, mkdir_err = run_command(f'mkdir -p {LOG_DIR} {MANIFEST_DIR}')
if mkdir_err:
    print(f'[WARN] mkdir: {mkdir_err}')

force_flag = '--force-recompute' if FORCE_RECOMPUTE else ''

cmd = (
    f'sbatch --job-name ego_prepare -c 4 --mem=32G -t 02:00:00 '
    f'--export=ALL,CONDA_ENV_NAME={CONDA_ENV} '
    f'--chdir {PIPELINE_WORKDIR} '
    f'--output {LOG_DIR}/%x_%j.out --error {LOG_DIR}/%x_%j.err '
    f'{sh_script} {PIPELINE_WORKDIR} {PREPARE_SCRIPT} '
    f'--data-root {DATA_ROOT} '
    f'--figures-root {FIGURES_ROOT} '
    f'--output-dir {MANIFEST_DIR} '
    f'--categories CSplus CSminus '
    f'{force_flag}'
)

output, error = run_command(cmd)
prepare_job_id = None
if error:
    print(f'[ERROR] {error}')
else:
    print(f'[INFO] {output}')
    match = re.search(r'Submitted batch job (\d+)', output)
    if match:
        prepare_job_id = match.group(1)
        print(f'[INFO] Prepare Job ID: {prepare_job_id}')
        print(f'[INFO] Log .out: {to_local(LOG_DIR)}/ego_prepare_{prepare_job_id}.out')
        print(f'[INFO] Log .err: {to_local(LOG_DIR)}/ego_prepare_{prepare_job_id}.err')

[INFO] Submitted batch job 29149498
[INFO] Prepare Job ID: 29149498
[INFO] Log .out: /Volumes/adam-lab/qixin.yang/ClusterCode/miniVI_Renana_Pipeline/miniVI_PlaceCell_analysis_V4/logs/ego_prepare_29149498.out
[INFO] Log .err: /Volumes/adam-lab/qixin.yang/ClusterCode/miniVI_Renana_Pipeline/miniVI_PlaceCell_analysis_V4/logs/ego_prepare_29149498.err


In [6]:
# Wait for prepare job to finish
if prepare_job_id:
    wait_for_jobs([prepare_job_id], poll_interval=30)
else:
    print('[INFO] No prepare job submitted.')

[19:27:25] Jobs status:
  Completed: 1/1
  Pending:   0

All jobs completed successfully!


# Step 2: Submit Array Job — One Task Per Cell

Each SLURM array task processes one cell independently.
All tasks run in parallel across the cluster.

In [4]:
# Read manifest to get cell count
import json

manifest_output, manifest_error = run_command(f'cat {MANIFEST_DIR}/manifest.json')
if manifest_error:
    print(f'[ERROR] {manifest_error}')
    N_CELLS = 0
else:
    manifest = json.loads(manifest_output)
    N_CELLS = len(manifest)
    print(f'Total cells to analyze: {N_CELLS}')
    for cat in set(m['category'] for m in manifest):
        n = sum(1 for m in manifest if m['category'] == cat)
        print(f'  {cat}: {n}')

Total cells to analyze: 56
  CSplus: 12
  all-nonPLC: 36
  CSminus: 8


In [5]:
if N_CELLS == 0:
    print('[ERROR] No cells in manifest. Check Step 1.')
    array_job_id = None
else:
    cmd = (
        f'sbatch --job-name ego_cell '
        f'--array=0-{N_CELLS - 1} '
        f'-c {CELL_CPUS} --mem={CELL_MEM_GB}G '
        f'--export=ALL,CONDA_ENV_NAME={CONDA_ENV} '
        f'--chdir {PIPELINE_WORKDIR} '
        f'--output {LOG_DIR}/ego_cell_%A_%a.out '
        f'--error {LOG_DIR}/ego_cell_%A_%a.err '
        f'{sh_script} {PIPELINE_WORKDIR} {CELL_SCRIPT} '
        f'--output-dir {OUTPUT_DIR} '
        f'--manifest-dir {MANIFEST_DIR} '
        f'--data-root {DATA_ROOT} '
        f'--figures-root {FIGURES_ROOT} '
        f'--direction-mode {DIRECTION_MODE} '
        f'--spike-type {SPIKE_TYPE} '
        f'--n-surrogates {N_SURROGATES} '
        f'--n-jobs {CELL_CPUS} '
        f'--first-n-minutes {FIRST_N_MINUTES}'
    )

    output, error = run_command(cmd)
    array_job_id = None
    if error:
        print(f'[ERROR] {error}')
    else:
        print(f'[INFO] {output}')
        match = re.search(r'Submitted batch job (\d+)', output)
        if match:
            array_job_id = match.group(1)
            print(f'[INFO] Array Job ID: {array_job_id}')
            print(f'[INFO] {N_CELLS} tasks submitted (0 to {N_CELLS - 1})')
            print(f'[INFO] Each task: {CELL_CPUS} CPUs, {CELL_MEM_GB}GB memory')
            print(f'[INFO] Config: {DIRECTION_MODE} direction, {SPIKE_TYPE} spikes')
            print(f'[INFO] Log dir (local): {to_local(LOG_DIR)}/')
            print(f'[INFO] Log pattern: ego_cell_{array_job_id}_<task_id>.out/.err')

[INFO] Submitted batch job 29535514
[INFO] Array Job ID: 29535514
[INFO] 56 tasks submitted (0 to 55)
[INFO] Each task: 10 CPUs, 32GB memory
[INFO] Config: travel direction, all_spike spikes
[INFO] Log dir (local): /Volumes/adam-lab/qixin.yang/ClusterCode/miniVI_Renana_Pipeline/miniVI_PlaceCell_analysis_V4/logs/
[INFO] Log pattern: ego_cell_29535514_<task_id>.out/.err


In [ ]:
# Wait for all array tasks to complete
if array_job_id:
    # For array jobs, monitor the main job ID
    wait_for_jobs([array_job_id], poll_interval=30)
else:
    print('[INFO] No array job submitted.')

# Step 3: Optional Cluster Summaries

Step 2 already writes the EB tuning/shuffle decision for each cell (`pass_95`, `pass_99`, `empirical_p`, MRL, and fit parameters) into `per_cell_results/*.npz`. For the current trace-free egocentric-only bundle, you can skip Step 3 entirely and later copy the results back locally for `LocalFig_egocentric_head_three_spike_any100_analysis_full.ipynb`.

## 3a — Per-cell summary plots
Do not run this with the default egocentric-only bundle. This plot script needs trace-bearing runtime data.

## 3b — Summary statistics
This reads only per-cell `.npz` results and remains usable if you want quick cluster-side pass-count and MRL summaries.


In [16]:
# Step 3a: Per-cell summary plots
# Optional only for trace-bearing bundles. The default egocentric-only export excludes traces,
# so leave this cell unrun for the current EB shuffle workflow.


cmd = (
    f'sbatch --job-name ego_plot -c 4 --mem=64G -t 04:00:00 '
    f'--export=ALL,CONDA_ENV_NAME={CONDA_ENV} '
    f'--chdir {PIPELINE_WORKDIR} '
    f'--output {LOG_DIR}/%x_%j.out --error {LOG_DIR}/%x_%j.err '
    f'{sh_script} {PIPELINE_WORKDIR} {PLOT_SCRIPT} '
    f'--output-dir {OUTPUT_DIR} '
    f'--manifest-dir {MANIFEST_DIR} '
    f'--data-root {DATA_ROOT} '
    f'--figures-root {FIGURES_ROOT} '
    f'--direction-mode {DIRECTION_MODE} '
    f'--first-n-minutes {FIRST_N_MINUTES} '
    f'--save-formats svg png'
)

output, error = run_command(cmd)
plot_job_id = None
if error:
    print(f'[ERROR] {error}')
else:
    print(f'[INFO] {output}')
    match = re.search(r'Submitted batch job (\d+)', output)
    if match:
        plot_job_id = match.group(1)
        print(f'[INFO] Plot Job ID: {plot_job_id}')
        print(f'[INFO] Log .out: {to_local(LOG_DIR)}/ego_plot_{plot_job_id}.out')
        print(f'[INFO] Log .err: {to_local(LOG_DIR)}/ego_plot_{plot_job_id}.err')
        print(f'[INFO] Figures dir (local): {to_local(OUTPUT_DIR)}/per_cell_summary/')

[INFO] Submitted batch job 28153847
[INFO] Plot Job ID: 28153847
[INFO] Log .out: /Volumes/adam-lab/qixin.yang/ClusterCode/miniVI_Renana_Pipeline/miniVI_PlaceCell_analysis_V4/logs/ego_plot_28153847.out
[INFO] Log .err: /Volumes/adam-lab/qixin.yang/ClusterCode/miniVI_Renana_Pipeline/miniVI_PlaceCell_analysis_V4/logs/ego_plot_28153847.err
[INFO] Figures dir (local): /Volumes/adam-lab/qixin.yang/ClusterCode/miniVI_Renana_Pipeline/miniVI_PlaceCell_analysis_V4/figures/CKII_pooled/egocentric_tuning_carpenter/head_all_spike/per_cell_summary/


In [17]:
# Wait for per-cell plots to finish
if plot_job_id:
    wait_for_jobs([plot_job_id], poll_interval=60)
else:
    print('[INFO] No plot job submitted.')

[22:28:40] Jobs status:
  Completed: 1/1
  Pending:   0

All jobs completed successfully!


In [18]:
# Step 3b: Summary statistics (CSplus vs CSminus)
# Reads .npz files directly — no aggregation step needed.
# Also saves a combined egocentric_tuning_summary.csv.
# Can run independently of Step 2 and 3a.

cmd = (
    f'sbatch --job-name ego_stats -c 1 --mem=4G -t 00:10:00 '
    f'--export=ALL,CONDA_ENV_NAME={CONDA_ENV} '
    f'--chdir {PIPELINE_WORKDIR} '
    f'--output {LOG_DIR}/%x_%j.out --error {LOG_DIR}/%x_%j.err '
    f'{sh_script} {PIPELINE_WORKDIR} {STATS_SCRIPT} '
    f'--output-dir {OUTPUT_DIR} '
    f'--manifest-dir {MANIFEST_DIR} '
    f'--categories CSplus CSminus '
    f'--save-formats svg png'
)

output, error = run_command(cmd)
stats_job_id = None
if error:
    print(f'[ERROR] {error}')
else:
    print(f'[INFO] {output}')
    match = re.search(r'Submitted batch job (\d+)', output)
    if match:
        stats_job_id = match.group(1)
        print(f'[INFO] Stats Job ID: {stats_job_id}')
        print(f'[INFO] Log .out: {to_local(LOG_DIR)}/ego_stats_{stats_job_id}.out')
        print(f'[INFO] Log .err: {to_local(LOG_DIR)}/ego_stats_{stats_job_id}.err')
        print(f'[INFO] Stats dir (local): {to_local(OUTPUT_DIR)}/summary_stats/')

[INFO] Submitted batch job 28154050
[INFO] Stats Job ID: 28154050
[INFO] Log .out: /Volumes/adam-lab/qixin.yang/ClusterCode/miniVI_Renana_Pipeline/miniVI_PlaceCell_analysis_V4/logs/ego_stats_28154050.out
[INFO] Log .err: /Volumes/adam-lab/qixin.yang/ClusterCode/miniVI_Renana_Pipeline/miniVI_PlaceCell_analysis_V4/logs/ego_stats_28154050.err
[INFO] Stats dir (local): /Volumes/adam-lab/qixin.yang/ClusterCode/miniVI_Renana_Pipeline/miniVI_PlaceCell_analysis_V4/figures/CKII_pooled/egocentric_tuning_carpenter/head_all_spike/summary_stats/


In [19]:
# Wait for stats job and check results
if stats_job_id:
    wait_for_jobs([stats_job_id], poll_interval=15)

    # Print stats log
    log_path = f'{LOG_DIR}/ego_stats_{stats_job_id}.out'
    output, error = run_command(f'cat {log_path}')
    if output:
        print(output)
    if error:
        print(f'[ERROR] {error}')
else:
    print('[INFO] No stats job submitted.')

[22:29:08] Jobs status:
  Completed: 1/1
  Pending:   0

All jobs completed successfully!
Starting Python: miniVI_PlaceCell_analysis_V4/notebooks_HPC/run_egocentric_summary_stats.py
Workdir: /ems/elsc-labs/adam-y/qixin.yang/ClusterCode/miniVI_Renana_Pipeline
Args: --output-dir /ems/elsc-labs/adam-y/qixin.yang/ClusterCode/miniVI_Renana_Pipeline/miniVI_PlaceCell_analysis_V4/figures/CKII_pooled/egocentric_tuning_carpenter/head_all_spike --manifest-dir /ems/elsc-labs/adam-y/qixin.yang/ClusterCode/miniVI_Renana_Pipeline/miniVI_PlaceCell_analysis_V4/figures/CKII_pooled/egocentric_tuning_carpenter --categories CSplus CSminus all-nonPLC --save-formats svg png
Current working directory: /ems/elsc-labs/adam-y/qixin.yang/ClusterCode/miniVI_Renana_Pipeline
Python executable: /ems/elsc-labs/adam-y/qixin.yang/anaconda3/envs/adamlab_pipeline/bin/python
Python 3.11.13
Loaded manifest: 56 cells
NPZ results: 45 success (in requested categories), 10 skipped, 1 missing
Total rows for analysis: 45

=== Pas

In [20]:
# Check per-cell plot outputs
output, error = run_command(
    f'echo "=== Per-cell plot folders ===" && '
    f'ls -la {OUTPUT_DIR}/per_cell_summary/ 2>/dev/null && '
    f'echo "" && '
    f'for d in {OUTPUT_DIR}/per_cell_summary/*/; do '
    f'  name=$(basename "$d"); '
    f'  count=$(ls "$d"/*.svg 2>/dev/null | wc -l); '
    f'  echo "  $name: $count SVG files"; '
    f'done && '
    f'echo "" && '
    f'echo "=== Summary stats ===" && '
    f'ls -la {OUTPUT_DIR}/summary_stats/ 2>/dev/null'
)
print(output)
if error:
    print(f'[WARN] {error}')

=== Per-cell plot folders ===
total 376
drwxr-xr-x. 5 qixin.yang adam-lab   213 Mar 24 22:25 .
drwxr-xr-x. 5 qixin.yang adam-lab   204 Mar 24 22:28 ..
drwxr-xr-x. 2 qixin.yang adam-lab  3756 Mar 24 22:25 all-nonPLC
drwxr-xr-x. 2 qixin.yang adam-lab  1008 Mar 24 22:23 CSminus
drwxr-xr-x. 2 qixin.yang adam-lab  1732 Mar 24 22:24 CSplus
-rwxrwx---. 1 qixin.yang adam-lab 10244 Mar 24 22:27 .DS_Store
-rw-r--r--. 1 qixin.yang adam-lab 25565 Mar 24 22:25 egocentric_per_cell_plot_manifest.csv
-rw-r--r--. 1 qixin.yang adam-lab   696 Mar 24 22:25 egocentric_per_cell_plot_skipped.csv

  all-nonPLC: 26 SVG files
  CSminus: 7 SVG files
  CSplus: 12 SVG files

=== Summary stats ===
total 424
drwxr-xr-x. 2 qixin.yang adam-lab   336 Mar 24 22:28 .
drwxr-xr-x. 5 qixin.yang adam-lab   204 Mar 24 22:28 ..
-rw-r--r--. 1 qixin.yang adam-lab  8283 Mar 24 22:28 egocentric_pass_counts_all_categories.png
-rw-r--r--. 1 qixin.yang adam-lab  8902 Mar 24 22:28 egocentric_pass_counts_all_categories.svg
-rw-r--r--. 